# 02 Strata

Classify every sampling unit into forest type x seral proxy x density proxy (cover as an attribute), collapse cells below the minimum area, and produce the cell table with acres. Includes the height-to-QMD calibration step that turns the placeholder breaks in `config.yaml` into defensible ones once plots with both QMD and LiDAR height are available.

In [ ]:
import sys, os
print(sys.executable)
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
from src.io import load_config, get_logger
from src import strata, qa
cfg = load_config()
log = get_logger("02_strata")
log.info(f"Project: {cfg['project']['name']} | synthetic={cfg['run']['synthetic']} | freeze={cfg['run']['freeze']}")
P = Path(cfg["paths"]["processed"]); O = Path(cfg["paths"]["outputs"]); P.mkdir(parents=True, exist_ok=True); O.mkdir(exist_ok=True)

In [ ]:
frame = pd.read_parquet(P / "frame.parquet")
log.info(f"Frame: {len(frame):,} units")

## Calibrate height breaks to the QMD definition (when calibration plots exist)

The standard defines seral by QMD (5 and 25 in). We need the p95 height that corresponds to those QMDs per forest type. Fit on Lake Tahoe West validation plots (and FIA if coordinates are available), then write the breaks back to `config.yaml` by hand and record the fit in `docs/DESIGN_SUMMARY.md`. Skipped on the synthetic run.

In [ ]:
calib = cfg["sources"]["ltw_plots"]
if (str(calib).startswith("http") or Path(calib).exists()) and not cfg["run"]["synthetic"]:
    import geopandas as gpd, rasterio
    from src.layers import read_layer, read_raster
    plots = read_layer(cfg["sources"]["ltw_plots"], cfg, log=log)
    arr, tr, _ = read_raster(cfg["sources"]["p95_height_30m"], cfg, bbox=tuple(plots.total_bounds), log=log)
    rr, cc = rasterio.transform.rowcol(tr, plots.geometry.x.values, plots.geometry.y.values)
    plots["p95_height_m"] = arr[np.clip(rr, 0, arr.shape[0]-1), np.clip(cc, 0, arr.shape[1]-1)]
    for ftype, sub in plots.groupby("forest_type"):
        # monotone fit: height as a function of QMD, then invert at 5 and 25 in
        coef = np.polyfit(sub["qmd_in"], sub["p95_height_m"], 1)
        breaks = [float(np.polyval(coef, q)) for q in cfg["threshold"]["qmd_breaks_in"]]
        log.info(f"{ftype}: n={len(sub)} height = {coef[0]:.2f}*QMD + {coef[1]:.2f}; breaks at QMD 5/25 in -> {breaks[0]:.1f} / {breaks[1]:.1f} m")
else:
    log.info("No calibration plots yet; using placeholder height breaks from config.yaml")

## Classify and collapse

In [ ]:
classified = strata.classify_cells(frame, cfg)
cells_raw = classified.groupby(["cell_id", "forest_type", "seral_class", "density_class"], as_index=False)["acres"].sum()
log.info(f"Raw cells: {len(cells_raw)}; below {cfg['strata']['min_cell_acres']} ac: {(cells_raw['acres'] < cfg['strata']['min_cell_acres']).sum()}")
classified, collapse_log = strata.collapse_small_cells(classified, cfg, log)
collapse_log.to_csv(O / "cell_collapse_log.csv", index=False)
cells = classified.groupby(["cell_id", "forest_type", "seral_class", "density_class"], as_index=False)["acres"].sum().sort_values("cell_id")
cells["share_of_type"] = cells["acres"] / cells.groupby("forest_type")["acres"].transform("sum")
cells.to_csv(O / "cells.csv", index=False)
classified.to_parquet(P / "frame_classified.parquet", index=False)
log.info(f"Populated cells after collapse: {len(cells)}")
cells

## Cover distribution by cell (attribute check; decides whether cover becomes a fourth axis)

In [ ]:
xt = pd.crosstab(classified["cell_id"], classified["cover_class"], values=classified["acres"], aggfunc="sum").fillna(0).round()
xt.to_csv(O / "cells_by_cover.csv")
xt